First, we want a way of storing k,v for later use

Next, load methods for prompts for the model

In [1]:
"""
Install the Google AI Python SDK

$ pip install google-generativeai

See the getting started guide for more information:
https://ai.google.dev/gemini-api/docs/get-started/python
"""

import os
import json
import google.generativeai as genai
from google.generativeai.types import HarmCategory, HarmBlockThreshold

genai.configure(api_key="XXXXX")

# Create the model
# See https://ai.google.dev/api/python/google/generativeai/GenerativeModel
generation_config = {
  "temperature": 2,
  "top_p": 0.95,
  "top_k": 64,
  "max_output_tokens": 8192,
  "response_mime_type": "text/plain",
}

model = genai.GenerativeModel(
  model_name="gemini-2.0-flash",
  generation_config=generation_config,

  # See https://ai.google.dev/gemini-api/docs/safety-settings
)
def gemini_insertion():
    # Ensure data.json exists
    if not os.path.exists("data.json"):
        with open("data.json", "w") as f:
            json.dump({}, f)
            
    data = {}
    with open("data.json", "r") as f:
        data = json.load(f)

    print(f"Currently {len(data)} names in data.json")

    response = model.generate_content([
        "Your role is to generate a single name and nothing more that is the name for a robot dog. "
        "This robot dog is fun, caring, kind, adventurous, and funny. It is powered by AI and is supposed to be the robot dog of the future. "
        "It is inspired by the robot dog from the Jetsons, and will serve humanity for the better. "
        "This name should be creative, special, and catchy. Please provide as unique of an answer as possible—one which people will enjoy. "
        "Please only provide one name.\n"
        "In addition, this name MUST NOT be any of the following names: " + ", ".join(data.keys()) + "\n"
        "input: Please generate just a creative name and nothing more for this robot dog",
        "output: "
    ])

    new_name = response.text.strip()
    data[new_name] = 0

    with open("data.json", "w") as f:
        json.dump(data, f, indent=2)

    print(f"Generated name: {new_name}")
    return new_name

C:\Users\diggy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Bradley-Terry Model

In [2]:
import os
import json
import time
import random
from itertools import combinations

def filter_down():
    # Ensure data.json exists
    if not os.path.exists("data.json"):
        with open("data.json", "w") as f:
            json.dump({}, f)

    with open("data.json", "r") as f:
        data = json.load(f)

    items = list(data.keys())

    # Save permanent full item list (only once)
    if not os.path.exists("item_list.json"):
        with open("item_list.json", "w") as f:
            json.dump(items, f)

    with open("item_list.json", "r") as f:
        stable_items = json.load(f)

    item_to_index = {item: idx for idx, item in enumerate(stable_items)}

    # Ensure ranks.json exists
    if not os.path.exists("ranks.json"):
        with open("ranks.json", "w") as f:
            json.dump([], f)

    with open("ranks.json", "r") as f:
        try:
            pairwise_results = json.load(f)
        except json.JSONDecodeError:
            pairwise_results = []

    print(f"Loaded {len(pairwise_results)} existing comparisons.")

    # Random unique pairs
    all_pairs = list(combinations(items, 2))
    random.shuffle(all_pairs)

    for a, b in all_pairs:
        prompt = (
            "Your role is to evaluate how accurately and well perceived the names are for a robot dog. "
            "You shall answer with a 1 if you prefer name 1, or 2 if you prefer name 2. "
            "This robot dog is fun, caring, kind, adventurous, and funny. It is powered by AI and inspired by Astro from the Jetsons. "
            "The look of the dog is the Unitree Go 2"
            "The name should be creative, special, and catchy. Please provide a single number only (1 or 2).\n\n"
            f"Which do you prefer:\n1) {a}\n2) {b}\n\nOnly respond with 1 or 2."
        )

        print(prompt)

        response = model.generate_content(prompt)
        choice = response.text.strip()

        if choice == '1':
            pairwise_results.append([item_to_index[a], item_to_index[b]])
        elif choice == '2':
            pairwise_results.append([item_to_index[b], item_to_index[a]])
        else:
            print("Invalid response, skipping.")
            continue

        print(f"Model chose: {choice} → {a if choice == '1' else b}")
        time.sleep(4)

        with open("ranks.json", "w") as f:
            json.dump(pairwise_results, f)

    print(f"Saved total of {len(pairwise_results)} comparisons.")


In [3]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os
import json
import choix

def selectfin():
    # Load stable item list
    if not os.path.exists("item_list.json"):
        print("Missing item_list.json.")
        return

    with open("item_list.json", "r") as f:
        items = json.load(f)

    n_items = len(items)
    index_to_item = {i: item for i, item in enumerate(items)}
    leader_count = {item: 0 for item in items}

    if not os.path.exists("ranks.json"):
        print("Missing ranks.json.")
        return

    with open("ranks.json", "r") as f:
        pairwise_results = json.load(f)

    os.makedirs("rank_graphs", exist_ok=True)
    prev_ranking = []

    # Create a consistent color map for all items (use a big enough colormap)
    cmap = cm.get_cmap('tab20', n_items)
    # Map each item to a fixed color
    name_to_color = {item: cmap(i) for i, item in enumerate(items)}

    for i in range(1, len(pairwise_results) + 1):
        current_results = pairwise_results[:i]
        try:
            abilities = choix.ilsr_pairwise(n_items=n_items, data=current_results, alpha=1e-3)
        except Exception as e:
            print(f"Iteration {i} — error: {e}")
            continue

        ranking = sorted(enumerate(abilities), key=lambda x: -x[1])
        current_ranking = [index_to_item[idx] for idx, _ in ranking]
        top_item = current_ranking[0]
        leader_count[top_item] += 1

        if current_ranking != prev_ranking:
            prev_ranking = current_ranking.copy()
            top_10 = ranking[:10]
            labels = [index_to_item[idx] for idx, _ in top_10]
            scores = [score for _, score in top_10]

            # Use the fixed color for each label
            bar_colors = [name_to_color[name] for name in labels]

            plt.figure(figsize=(10, 6))
            bars = plt.bar(labels, scores, color=bar_colors)
            plt.xlabel("Name")
            plt.ylabel("Score")
            plt.title(f"Ranking at Iteration {i}")
            plt.ylim(min(scores) - 0.1, max(scores) + 0.1)

            for bar, score in zip(bars, scores):
                plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                         f"{score:.2f}", ha='center', va='bottom')

            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.savefig(f"rank_graphs/rank_{i:03d}.png")
            plt.close()
            print(f"Saved rank_graphs/rank_{i:03d}.png")


    # Final scores + update data.json without trimming
    final_abilities = choix.ilsr_pairwise(n_items=n_items, data=pairwise_results, alpha=1e-3)
    final_ranking = sorted(
        [(index_to_item[i], score) for i, score in enumerate(final_abilities)],
        key=lambda x: -x[1]
    )

    final_scores = {name: round(score, 3) for name, score in final_ranking}

    if os.path.exists("data.json"):
        with open("data.json", "r") as f:
            data = json.load(f)
    else:
        data = {}

    for name, score in final_scores.items():
        data[name] = score

    with open("data.json", "w") as f:
        json.dump(data, f, indent=2)

    with open("final.json", "w") as f:
        json.dump(leader_count, f, indent=2)

    print("\nFinal Top 10 Ranking:")
    for i, (name, score) in enumerate(final_ranking[:10], 1):
        print(f"{i}. {name} — {score:.3f}")


In [4]:
import cv2
import os
from natsort import natsorted  # to sort images in natural order

def images_to_video(image_folder="rank_graphs", output_filename="rankings_video.mp4", fps=40):
    # Get sorted list of .png files
    images = [img for img in os.listdir(image_folder) if img.endswith(".png")]
    images = natsorted(images)  # Ensure proper order like 1, 2, 10

    if not images:
        print("No PNG images found in the folder.")
        return

    # Read the first image to get frame size
    first_img_path = os.path.join(image_folder, images[0])
    frame = cv2.imread(first_img_path)
    height, width, layers = frame.shape

    # Define the codec and create VideoWriter object
    video = cv2.VideoWriter(output_filename, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    for image in images:
        img_path = os.path.join(image_folder, image)
        frame = cv2.imread(img_path)
        if frame is None:
            print(f"Skipping unreadable image: {img_path}")
            continue
        video.write(frame)

    video.release()
    print(f"Video saved as {output_filename}")

In [209]:
for i in range(10):
    gemini_insertion()
    time.sleep(4)

Currently 6 names in data.json
Generated name: AstroBot
Currently 7 names in data.json
Generated name: DigiDog
Currently 8 names in data.json
Generated name: Robo Wagsly
Currently 9 names in data.json
Generated name: Fetch.ai
Currently 10 names in data.json
Generated name: Clip.
Currently 11 names in data.json
Generated name: Jaxx
Currently 12 names in data.json
Generated name: Gizmotron


KeyboardInterrupt: 

In [5]:
filter_down()

Loaded 3459 existing comparisons.
Your role is to evaluate how accurately and well perceived the names are for a robot dog. You shall answer with a 1 if you prefer name 1, or 2 if you prefer name 2. This robot dog is fun, caring, kind, adventurous, and funny. It is powered by AI and inspired by Astro from the Jetsons. The look of the dog is the Unitree Go 2The name should be creative, special, and catchy. Please provide a single number only (1 or 2).

Which do you prefer:
1) ZipChip
2) FidoLogic

Only respond with 1 or 2.
Model chose: 2 → FidoLogic
Your role is to evaluate how accurately and well perceived the names are for a robot dog. You shall answer with a 1 if you prefer name 1, or 2 if you prefer name 2. This robot dog is fun, caring, kind, adventurous, and funny. It is powered by AI and inspired by Astro from the Jetsons. The look of the dog is the Unitree Go 2The name should be creative, special, and catchy. Please provide a single number only (1 or 2).

Which do you prefer:
1)

KeyboardInterrupt: 

In [7]:
selectfin()

C:\Users\diggy\AppData\Local\Temp\ipykernel_40968\3377118309.py:31: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('tab20', n_items)


Saved rank_graphs/rank_001.png
Saved rank_graphs/rank_002.png
Saved rank_graphs/rank_003.png
Saved rank_graphs/rank_004.png
Saved rank_graphs/rank_005.png
Saved rank_graphs/rank_006.png
Saved rank_graphs/rank_007.png
Saved rank_graphs/rank_008.png
Saved rank_graphs/rank_009.png
Saved rank_graphs/rank_010.png
Saved rank_graphs/rank_011.png
Saved rank_graphs/rank_012.png
Saved rank_graphs/rank_013.png
Saved rank_graphs/rank_014.png
Saved rank_graphs/rank_015.png
Saved rank_graphs/rank_016.png
Saved rank_graphs/rank_017.png
Saved rank_graphs/rank_018.png
Saved rank_graphs/rank_019.png
Saved rank_graphs/rank_020.png
Saved rank_graphs/rank_021.png
Saved rank_graphs/rank_022.png
Saved rank_graphs/rank_023.png
Saved rank_graphs/rank_024.png
Saved rank_graphs/rank_025.png
Saved rank_graphs/rank_026.png
Saved rank_graphs/rank_027.png
Saved rank_graphs/rank_028.png
Saved rank_graphs/rank_029.png
Saved rank_graphs/rank_030.png
Saved rank_graphs/rank_031.png
Saved rank_graphs/rank_032.png
Saved ra

In [8]:
# Run the function
images_to_video("rank_graphs", "rankings_video.mp4", 60)

Video saved as rankings_video.mp4
